# Sesión 2 · Validación cruzada y outliers

**Duración estimada:** 2h30  
**Objetivo:** aprender a evaluar mejor un modelo de regresión y entender cómo afectan los outliers.

## Objetivos de aprendizaje

Al terminar este notebook deberías poder:
- Explicar por qué una sola partición train/test puede no ser suficiente.
- Usar `cross_val_score` en un problema de regresión.
- Detectar outliers con una regla simple.
- Comparar resultados con y sin outliers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 1. Repaso rápido

En la sesión anterior usamos una única división train/test. Eso está bien para empezar, pero tiene una limitación: el resultado depende del corte que haya tocado.

In [ ]:
data = load_diabetes(as_frame=True)
df = data.frame.copy()

X = df.drop(columns='target')
y = df['target']

## 2. Un mismo modelo puede dar resultados algo distintos

Vamos a entrenar varias veces cambiando el `random_state` del `train_test_split`.

In [ ]:
resultados = []

for seed in [1, 7, 21, 42, 99]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    resultados.append({
        'seed': seed,
        'R2': r2_score(y_test, pred),
        'MAE': mean_absolute_error(y_test, pred)
    })

pd.DataFrame(resultados)

## Idea clave

El modelo es el mismo, pero el score cambia algo según la partición. Por eso usamos **validación cruzada**.

## 3. Validación cruzada

En K-Fold dividimos los datos en varias partes (*folds*). Entrenamos varias veces y cada parte actúa como test una vez.
Así obtenemos una evaluación más estable.

In [ ]:
model = LinearRegression()
cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')

print('Scores R2 por fold:', np.round(cv_scores, 3))
print('Media:', round(cv_scores.mean(), 3))
print('Desv. típica:', round(cv_scores.std(), 3))

## Ejercicio 1

Ejecuta ahora validación cruzada usando como métrica el error absoluto medio.
Pista: en scikit-learn algunos errores aparecen con signo negativo.

In [ ]:
# Tu código aquí

### Solución 1

In [ ]:
cv_mae = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
print('MAE por fold:', np.round(-cv_mae, 2))
print('MAE medio  :', round((-cv_mae).mean(), 2))

## 4. ¿Qué es un outlier?

Un outlier es un dato muy alejado del comportamiento habitual. Puede ser:
- un error de medida;
- un caso raro pero real;
- una observación extrema que afecta mucho al modelo.

Para verlo mejor, vamos a crear un dataset pequeño y luego añadir algunos valores extremos.

In [ ]:
X_base, y_base = make_regression(
    n_samples=120,
    n_features=1,
    noise=15,
    random_state=10
)

X_out = X_base.copy()
y_out = y_base.copy()

X_out[:5] = np.array([[2.8], [3.0], [3.2], [3.4], [3.6]])
y_out[:5] = np.array([280, 310, 330, 350, 370])

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_base, y_base, alpha=0.7, label='Datos normales')
plt.title('Dataset sin outliers artificiales')
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_out, y_out, alpha=0.7, label='Con outliers')
plt.title('Dataset con outliers artificiales')
plt.show()

## 5. Comparar una regresión con y sin outliers

In [ ]:
model_base = LinearRegression().fit(X_base, y_base)
model_out = LinearRegression().fit(X_out, y_out)

x_line = np.linspace(min(X_out.min(), X_base.min()), max(X_out.max(), X_base.max()), 100).reshape(-1, 1)

plt.figure(figsize=(8, 5))
plt.scatter(X_base, y_base, alpha=0.5, label='Sin outliers')
plt.plot(x_line, model_base.predict(x_line), color='green', label='Recta sin outliers')
plt.scatter(X_out[:5], y_out[:5], color='red', s=60, label='Outliers')
plt.plot(x_line, model_out.predict(x_line), color='orange', label='Recta con outliers')
plt.legend()
plt.title('Efecto visual de los outliers')
plt.show()

## Ejercicio 2

Describe con tus palabras qué le ha pasado a la recta al añadir outliers.
¿Dirías que ahora representa mejor o peor la mayoría de los datos?

## 6. Detectar outliers con IQR

Una regla simple es usar el rango intercuartílico:
- calculamos Q1 y Q3;
- IQR = Q3 - Q1;
- marcamos como outliers valores fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]`.

In [ ]:
df_out = pd.DataFrame({'x': X_out.ravel(), 'y': y_out})

q1 = df_out['y'].quantile(0.25)
q3 = df_out['y'].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask_no_outliers = (df_out['y'] >= lower) & (df_out['y'] <= upper)

print('Límite inferior:', round(lower, 2))
print('Límite superior:', round(upper, 2))
print('Filas sin outliers:', mask_no_outliers.sum())
print('Filas detectadas como outlier:', (~mask_no_outliers).sum())

## Ejercicio 3

Crea `df_limpio` filtrando las filas sin outliers y vuelve a entrenar la regresión.
Después compara la pendiente o la visualización con el modelo anterior.

In [ ]:
# Tu código aquí

### Solución 3

In [ ]:
df_limpio = df_out[mask_no_outliers].copy()

X_clean = df_limpio[['x']]
y_clean = df_limpio['y']

model_clean = LinearRegression().fit(X_clean, y_clean)

plt.figure(figsize=(8, 5))
plt.scatter(df_out['x'], df_out['y'], alpha=0.4, label='Original con outliers')
plt.scatter(df_limpio['x'], df_limpio['y'], alpha=0.7, label='Sin outliers')
plt.plot(x_line, model_out.predict(x_line), color='orange', label='Modelo con outliers')
plt.plot(x_line, model_clean.predict(x_line), color='green', label='Modelo limpio')
plt.legend()
plt.title('Comparación final')
plt.show()

## Mini práctica guiada

1. Haz validación cruzada sobre el dataset de diabetes.
2. Guarda la media del R².
3. En el dataset sintético, detecta outliers con IQR.
4. Entrena un modelo antes y después de limpiar los outliers.
5. Escribe 3 conclusiones.

In [ ]:
# Resuelve aquí la mini práctica

## Cierre

Hoy has visto dos ideas centrales de ML:
- evaluar un modelo con una sola partición puede ser insuficiente;
- la calidad de los datos afecta mucho al resultado.